# Chapter 5: The Simulator and Disruption

This notebook brings the full pipeline to life. A simulator generates shipment events, publishes them to Confluent Cloud Kafka, and consumes them -- writing each event to both Neo4j and Snowflake Postgres. Mid-stream, a disruption fires: a warehouse goes inactive in Neo4j. From that point on, the consumer detects affected shipments, finds the shortest alternative route through the graph, and flags rerouted shipments in Snowflake Postgres.

This is the core value of the three-system architecture:

- **Confluent Kafka** streams events in real time
- **Neo4j** holds the route network and finds alternative paths instantly
- **Snowflake Postgres** records the operational impact -- which shipments were rerouted
  and via which alternative warehouse

## Before Running This Notebook

Chapters 2, 3 and 4 must have been run first. The Neo4j graph, Snowflake Postgres schema and Snowflake warehouse must all be in place before starting the simulator.

## 1. Install Dependencies

All versions are pinned for reproducibility. Run this cell once.

In [1]:
%pip install confluent-kafka==2.15.0 \
             neo4j==6.2.0 \
             numpy==2.2.6 \
             psycopg2-binary==2.9.12 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import numpy as np
import os
import psycopg2
import time
import uuid

from confluent_kafka import Consumer, KafkaError, Producer
from IPython.display import clear_output
from neo4j import GraphDatabase

## 3. Configuration

The simulator fires a disruption after `DISRUPTION_AFTER` events and restores the warehouse after `RESTORE_AFTER` events. Adjust these to control how long the disruption lasts relative to normal traffic.

The disrupted warehouse and the supplier-retailer pair used to demonstrate alternative routing match those verified in Chapter 2.

In [3]:
CONFLUENT_BOOTSTRAP_SERVERS = os.environ["CONFLUENT_BOOTSTRAP_SERVERS"]
CONFLUENT_API_KEY           = os.environ["CONFLUENT_API_KEY"]
CONFLUENT_API_SECRET        = os.environ["CONFLUENT_API_SECRET"]
TOPIC                       = "shipment-events"
GROUP_ID                    = "supply-chain-simulator"

NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

SNOWFLAKE_PG_HOST     = os.environ["SNOWFLAKE_PG_HOST"]
SNOWFLAKE_PG_USER     = os.environ["SNOWFLAKE_PG_USER"]
SNOWFLAKE_PG_PASSWORD = os.environ["SNOWFLAKE_PG_PASSWORD"]
SNOWFLAKE_PG_DBNAME   = os.environ["SNOWFLAKE_PG_DBNAME"]

RANDOM_SEED               = 42
DELAY_SECONDS             = 0.5     # pause between events
PRINT_EVERY               = 10      # refresh display every N events
DISRUPTION_INTERVAL_MIN   = 150     # min events between disruptions
DISRUPTION_INTERVAL_MAX   = 300     # max events between disruptions
DISRUPTION_DURATION_MIN   = 50      # min events a disruption lasts
DISRUPTION_DURATION_MAX   = 150     # max events a disruption lasts
FIRST_DISRUPTED_WAREHOUSE = "W003"  # matches Ch2 graph verification
N_EVENTS                  = -1      # set to a positive integer for testing, -1 to run continuously
MAX_RUNTIME_SECONDS       = 3600    # stop after 1 hour even if N_EVENTS = -1

SUPPLIER_IDS    = [f"S{i:03d}" for i in range(20)]
WAREHOUSE_IDS   = [f"W{i:03d}" for i in range(12)]
DIST_CENTER_IDS = [f"DC{i:03d}" for i in range(10)]
RETAILER_IDS    = [f"R{i:03d}" for i in range(30)]
STATUSES        = ["departed", "in_transit", "delayed", "delivered"]
STATUS_WEIGHTS  = [0.25, 0.45, 0.15, 0.15]

print("Configuration set.")

Configuration set.


## 4. Connect to Neo4j and Snowflake Postgres

In [4]:
neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_min_severity = "OFF",
)

print(neo4j_driver.verify_connectivity())  # None is expected
print("Connected to Neo4j Aura.")

None
Connected to Neo4j Aura.


In [5]:
pg_conn = psycopg2.connect(
    host     = SNOWFLAKE_PG_HOST,
    user     = SNOWFLAKE_PG_USER,
    password = SNOWFLAKE_PG_PASSWORD,
    dbname   = SNOWFLAKE_PG_DBNAME,
    sslmode  = "require",
    port     = 5432
)

pg_conn.autocommit = True
pg_cursor = pg_conn.cursor()
pg_cursor.execute("SELECT version()")
print("Connected to:", pg_cursor.fetchone()[0])

for table in ("shipments", "disruption_events"):
    pg_cursor.execute(f"TRUNCATE TABLE {table}")
    print(f"Table {table} truncated.")

Connected to: PostgreSQL 18.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 11.5.0 20240719 (Red Hat 11.5.0-14), 64-bit
Table shipments truncated.
Table disruption_events truncated.


## 5. Create the Kafka Producer and Consumer

The simulator uses both a producer and a consumer in the same notebook. The producer publishes events to the topic; the consumer reads them back and writes to Neo4j and Snowflake Postgres. Using a unique group ID ensures this consumer reads all messages
from the beginning of the topic on first run.

In [6]:
kafka_config = {
    "bootstrap.servers": CONFLUENT_BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms":   "PLAIN",
    "sasl.username":     CONFLUENT_API_KEY,
    "sasl.password":     CONFLUENT_API_SECRET,
    "log_level":         0,
}

producer = Producer(kafka_config)

consumer = Consumer({
    **kafka_config,
    "group.id":           GROUP_ID,
    "auto.offset.reset":  "earliest",
    "enable.auto.commit": True,
})
consumer.subscribe([TOPIC])

def delivery_report(err, msg):
    if err is not None:
        print(f"Delivery failed: {err}")

print("Producer and consumer created.")

Producer and consumer created.


## 6. Define the Event and Write Functions

The write functions extend those in Chapter 5b to support rerouting. When a shipment is rerouted, the Neo4j write updates the `VIA_WAREHOUSE` relationship to point to the alternative warehouse, and the Snowflake Postgres write sets `rerouted=True` and
records the alternative warehouse ID.

In [7]:
rng = np.random.default_rng(RANDOM_SEED)

def make_shipment_event():
    """Generate a single synthetic shipment event."""
    status = str(rng.choice(STATUSES, p=STATUS_WEIGHTS))
    return {
        "shipment_id":    str(uuid.uuid4()),
        "supplier_id":    str(rng.choice(SUPPLIER_IDS)),
        "warehouse_id":   str(rng.choice(WAREHOUSE_IDS)),
        "dist_center_id": str(rng.choice(DIST_CENTER_IDS)),
        "retailer_id":    str(rng.choice(RETAILER_IDS)),
        "status":         status,
        "timestamp":      time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "delay_minutes":  int(rng.integers(15, 240)) if status == "delayed" else 0,
    }

def write_to_neo4j(session, event, alt_warehouse_id=None):
    """Write a shipment event to Neo4j. If rerouted, update the warehouse relationship."""
    effective_warehouse = alt_warehouse_id or event["warehouse_id"]
    session.run("""
        MERGE (sh:Shipment {shipment_id: $shipment_id})
        SET sh.status        = $status,
            sh.delay_minutes = $delay_minutes,
            sh.timestamp     = $timestamp,
            sh.rerouted      = $rerouted

        WITH sh
        MATCH (s:Supplier            {id: $supplier_id})
        MATCH (w:Warehouse           {id: $effective_warehouse})
        MATCH (dc:DistributionCenter {id: $dist_center_id})
        MATCH (r:Retailer            {id: $retailer_id})

        MERGE (s)-[:HAS_SHIPMENT]->(sh)
        MERGE (sh)-[:VIA_WAREHOUSE]->(w)
        MERGE (sh)-[:VIA_DIST_CENTER]->(dc)
        MERGE (sh)-[:DESTINED_FOR]->(r)
    """,
        shipment_id         = event["shipment_id"],
        supplier_id         = event["supplier_id"],
        effective_warehouse = effective_warehouse,
        dist_center_id      = event["dist_center_id"],
        retailer_id         = event["retailer_id"],
        status              = event["status"],
        delay_minutes       = event["delay_minutes"],
        timestamp           = event["timestamp"],
        rerouted            = alt_warehouse_id is not None,
    )

def write_to_neo4j_with_retry(driver, event, alt_warehouse_id=None, retries=2):
    """Write a shipment event to Neo4j, reopening the session on connection failure."""
    for attempt in range(retries + 1):
        try:
            with driver.session() as session:
                write_to_neo4j(session, event, alt_warehouse_id)
            return
        except Exception:
            if attempt < retries:
                time.sleep(1)
            else:
                raise

def write_to_postgres(cursor, event, alt_warehouse_id=None):
    """Write a shipment event to Snowflake Postgres. Flag rerouted shipments."""
    cursor.execute("""
        INSERT INTO shipments (
            shipment_id, supplier_id, warehouse_id, dist_center_id,
            retailer_id, status, delay_minutes, rerouted, alt_warehouse_id
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (shipment_id) DO UPDATE
            SET rerouted         = EXCLUDED.rerouted,
                alt_warehouse_id = EXCLUDED.alt_warehouse_id,
                status           = EXCLUDED.status
    """,
        (
            event["shipment_id"],
            event["supplier_id"],
            event["warehouse_id"],
            event["dist_center_id"],
            event["retailer_id"],
            event["status"],
            event["delay_minutes"],
            alt_warehouse_id is not None,
            alt_warehouse_id,
        )
    )

def write_disruption_event(cursor, warehouse_id, event_type):
    """Log a disruption or restore event to the disruption_events table."""
    cursor.execute(
        """
        INSERT INTO disruption_events (warehouse_id, event_type)
        VALUES (%s, %s)
        """,
        (warehouse_id, event_type)
    )

print("Event and write functions defined.")

Event and write functions defined.


## 7. Define the Disruption Functions

Three functions manage the disruption lifecycle:

- `trigger_disruption()` marks the warehouse inactive in Neo4j.
- `find_alternative_route()` uses the variable-length path query from Chapter 2 to find the shortest path avoiding all inactive warehouses.
- `restore_warehouse()` removes the `active` flag to bring the warehouse back online.

The alternative route query uses `coalesce(n.active, true) = false` to handle nodes that have no `active` property -- treating absence as active. See the gotchas section for why this matters.

`DISRUPTED_WAREHOUSE` is fixed to a warehouse verified in Chapter 2 to have both active routes through it and at least one alternative. If you change this value, run the diagnostic queries from Chapter 2 first to confirm the new warehouse has sufficient network redundancy to make rerouting visible.

In [8]:
def trigger_disruption(session, warehouse_id):
    """Mark a warehouse as inactive in Neo4j."""
    session.run(
        "MATCH (w:Warehouse {id: $id}) SET w.active = false",
        id=warehouse_id
    )

def find_alternative_route(session, supplier_id, retailer_id):
    """Find the shortest path from supplier to retailer avoiding inactive warehouses.

    Returns the alternative warehouse ID if a rerouted path exists, or None if the
    supplier-retailer pair has no alternative route.
    """
    result = session.run("""
        MATCH (s:Supplier {id: $supplier_id}),
              (r:Retailer {id: $retailer_id})
        MATCH path = (s)-[:SHIPS_TO*1..6]->(r)
        WHERE NONE(
            n IN nodes(path)
            WHERE n:Warehouse AND coalesce(n.active, true) = false
        )
        RETURN [n IN nodes(path) | n.id] AS node_ids
        ORDER BY length(path) ASC
        LIMIT 1
    """,
        supplier_id = supplier_id,
        retailer_id = retailer_id,
    )
    record = result.single()
    if record is None:
        return None
    # The warehouse is always the second node in the path
    return record["node_ids"][1]

def restore_warehouse(session, warehouse_id):
    """Remove the active flag from a warehouse to bring it back online."""
    session.run(
        "MATCH (w:Warehouse {id: $id}) REMOVE w.active",
        id=warehouse_id
    )

print("Disruption functions defined.")

Disruption functions defined.


## 8. The Simulator Loop

The simulator produces and consumes events in a single loop. Disruptions fire randomly within the window defined by `DISRUPTION_INTERVAL_MIN` and `DISRUPTION_INTERVAL_MAX`. Each disruption lasts a random number of events between `DISRUPTION_DURATION_MIN` and `DISRUPTION_DURATION_MAX`, after which the warehouse is restored and the next disruption is scheduled for a different randomly chosen warehouse.

The display refreshes every `PRINT_EVERY` events showing:

- Total events processed
- Current pipeline phase: normal, disrupted or restored
- Latest event with rerouting details if applicable
- Status breakdown and rerouted count

The loop stops automatically after `MAX_RUNTIME_SECONDS` even if `N_EVENTS` is set to `-1`.

In [9]:
counter                   = 0
rerouted_count            = 0
counts                    = {s: 0 for s in STATUSES}
phase                     = "normal"
last_event                = None
last_reroute              = None
current_disruption        = FIRST_DISRUPTED_WAREHOUSE
next_disruption_warehouse = None

next_disruption = int(rng.integers(DISRUPTION_INTERVAL_MIN, DISRUPTION_INTERVAL_MAX + 1))
next_restore    = None
start_time      = time.time()

try:
    while (N_EVENTS == -1 or counter < N_EVENTS) and (time.time() - start_time < MAX_RUNTIME_SECONDS):

        # --- Produce one event ---
        event = make_shipment_event()
        producer.produce(
            topic    = TOPIC,
            key      = event["shipment_id"],
            value    = json.dumps(event),
            callback = delivery_report,
        )
        producer.poll(0)

        # --- Consume one event ---
        msg = consumer.poll(timeout=1.0)
        if msg is None or msg.error():
            time.sleep(DELAY_SECONDS)
            continue

        consumed = json.loads(msg.value().decode("utf-8"))
        alt_warehouse_id = None

        # --- Disruption lifecycle ---
        if phase == "normal" and counter >= next_disruption:
            with neo4j_driver.session() as session:
                trigger_disruption(session, current_disruption)
            write_disruption_event(pg_cursor, current_disruption, "disrupted")
            phase        = "disrupted"
            duration     = int(rng.integers(DISRUPTION_DURATION_MIN, DISRUPTION_DURATION_MAX + 1))
            next_restore = counter + duration

        elif phase == "disrupted" and counter >= next_restore:
            with neo4j_driver.session() as session:
                restore_warehouse(session, current_disruption)
            write_disruption_event(pg_cursor, current_disruption, "restored")
            phase           = "restored"
            gap             = int(rng.integers(DISRUPTION_INTERVAL_MIN, DISRUPTION_INTERVAL_MAX + 1))
            next_disruption = counter + gap
            next_restore    = None
            next_disruption_warehouse = str(rng.choice(WAREHOUSE_IDS))

        elif phase == "restored" and counter >= next_disruption:
            current_disruption = next_disruption_warehouse
            with neo4j_driver.session() as session:
                trigger_disruption(session, current_disruption)
            write_disruption_event(pg_cursor, current_disruption, "disrupted")
            phase        = "disrupted"
            duration     = int(rng.integers(DISRUPTION_DURATION_MIN, DISRUPTION_DURATION_MAX + 1))
            next_restore = counter + duration

        # --- Reroute if disrupted and shipment uses the disrupted warehouse ---
        if phase == "disrupted" and consumed["warehouse_id"] == current_disruption:
            with neo4j_driver.session() as session:
                alt_warehouse_id = find_alternative_route(
                    session,
                    consumed["supplier_id"],
                    consumed["retailer_id"],
                )
            if alt_warehouse_id:
                rerouted_count += 1
                last_reroute = {
                    "shipment_id": consumed["shipment_id"],
                    "original":    current_disruption,
                    "alternative": alt_warehouse_id,
                }

        write_to_neo4j_with_retry(neo4j_driver, consumed, alt_warehouse_id)
        write_to_postgres(pg_cursor, consumed, alt_warehouse_id)

        counts[consumed["status"]] += 1
        counter   += 1
        last_event = consumed

        # --- Refresh display ---
        if counter % PRINT_EVERY == 0 or counter == 1:
            clear_output(wait=True)
            phase_label = {
                "normal":    "Normal operations",
                "disrupted": f"DISRUPTION ACTIVE -- {current_disruption} offline",
                "restored":  f"Restored -- {current_disruption} back online",
            }[phase]

            print(f"Events processed : {counter}")
            print(f"Phase            : {phase_label}")
            print(f"Rerouted         : {rerouted_count}")
            print(f"Refresh every    : {PRINT_EVERY}\n")

            if last_event:
                icon = "!!" if last_event["status"] == "delayed" else "->"
                print("Latest event:")
                print(
                    f"  {icon} {last_event['shipment_id'][:8]}... "
                    f"{last_event['supplier_id']} -> "
                    f"{last_event['warehouse_id']} -> "
                    f"{last_event['dist_center_id']} -> "
                    f"{last_event['retailer_id']} "
                    f"[{last_event['status']}]"
                )

            if last_reroute:
                print()
                print("Last reroute:")
                print(f"  {last_reroute['shipment_id'][:8]}... "
                      f"{last_reroute['original']} -> {last_reroute['alternative']}")

            print("\nStatus breakdown:")
            for status, count in counts.items():
                print(f"  {status:<12} : {count}")

        time.sleep(DELAY_SECONDS)

except KeyboardInterrupt:
    print("\nSimulator stopped.")
finally:
    producer.flush(timeout=10)
    consumer.close()
    print(f"Total events processed : {counter}")
    print(f"Total rerouted         : {rerouted_count}")

Events processed : 5020
Phase            : DISRUPTION ACTIVE -- W007 offline
Rerouted         : 94
Refresh every    : 10

Latest event:
  -> 6323c7b7... S014 -> W003 -> DC002 -> R015 [delivered]

Last reroute:
  fde3ba0a... W007 -> W005

Status breakdown:
  departed     : 1257
  in_transit   : 2233
  delayed      : 811
  delivered    : 719

Simulator stopped.
Total events processed : 5023
Total rerouted         : 94


## 9. Verify

Run this cell after stopping the simulator to confirm both systems reflect the disruption correctly -- rerouted shipments should appear in both Neo4j and Snowflake Postgres.

In [10]:
# Verify Neo4j Aura
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_min_severity = "OFF",
)

with driver.session() as session:
    print("Neo4j -- Shipment nodes by status:")
    total = 0
    for status in STATUSES:
        result = session.run(
            "MATCH (sh:Shipment {status: $status}) RETURN count(sh) AS count",
            status=status
        )
        count = result.single()["count"]
        total += count
        print(f"  {status:<12} : {count}")
    print(f"\n  Total : {total}")

    result = session.run("""
        MATCH (sh:Shipment {rerouted: true})
        RETURN count(sh) AS count
    """)
    print(f"  Rerouted      : {result.single()['count']}")

driver.close()

Neo4j -- Shipment nodes by status:
  departed     : 1257
  in_transit   : 2235
  delayed      : 812
  delivered    : 719

  Total : 5023
  Rerouted      : 94


In [11]:
# Verify Snowflake Postgres
conn = psycopg2.connect(
    host     = SNOWFLAKE_PG_HOST,
    user     = SNOWFLAKE_PG_USER,
    password = SNOWFLAKE_PG_PASSWORD,
    dbname   = SNOWFLAKE_PG_DBNAME,
    sslmode  = "require",
    port     = 5432
)
conn.autocommit = True
cur = conn.cursor()

print("Snowflake Postgres -- shipments table by status:")
total = 0
for status in STATUSES:
    cur.execute(
        "SELECT COUNT(*) FROM shipments WHERE status = %s",
        (status,)
    )
    count = cur.fetchone()[0]
    total += count
    print(f"  {status:<12} : {count}")

cur.execute("SELECT COUNT(*) FROM shipments WHERE rerouted = TRUE")
print(f"\n  Total         : {total}")
print(f"  Rerouted      : {cur.fetchone()[0]}")

print("\nSnowflake Postgres -- disruption_events:")
cur.execute("""
    SELECT warehouse_id, event_type, recorded_at
    FROM disruption_events
    ORDER BY recorded_at
    LIMIT 10
""")
rows = cur.fetchall()
if rows:
    print(f"  {'Warehouse':<12} {'Event':<12} Timestamp")
    for row in rows:
        print(f"  {row[0]:<12} {row[1]:<12} {row[2]}")
else:
    print("  No disruption events recorded.")

cur.close()
conn.close()

Snowflake Postgres -- shipments table by status:
  departed     : 1257
  in_transit   : 2235
  delayed      : 812
  delivered    : 719

  Total         : 5023
  Rerouted      : 94

Snowflake Postgres -- disruption_events:
  Warehouse    Event        Timestamp
  W003         disrupted    2026-09-07 20:02:31.872035+00:00
  W003         restored     2026-09-07 20:03:44.466608+00:00
  W003         disrupted    2026-09-07 20:05:41.878910+00:00
  W003         restored     2026-09-07 20:06:24.182755+00:00
  W004         disrupted    2026-09-07 20:08:39.196101+00:00
  W004         restored     2026-09-07 20:09:36.938522+00:00
  W011         disrupted    2026-09-07 20:11:06.863915+00:00
  W011         restored     2026-09-07 20:11:38.065442+00:00
  W006         disrupted    2026-09-07 20:13:23.048598+00:00
  W006         restored     2026-09-07 20:14:03.947755+00:00
  W000         disrupted    2026-09-07 20:16:23.216858+00:00
  W000         restored     2026-09-07 20:17:45.104902+00:00
  W001  

## 10. Teardown

Close all connections. The data stays in both systems -- the Streamlit dashboards
in Chapter 7 read from Snowflake Postgres and the Snowflake warehouse to visualize
what the simulator produced.

In [12]:
pg_cursor.close()
pg_conn.close()
neo4j_driver.close()
print("All connections closed.")

All connections closed.
